# AWQ quantization

AWQ is a post-training, weight-only method. The MLSys paper observes that protecting "1% of salient weights" can greatly reduce quantization error.

Sources:
- Paper: https://arxiv.org/abs/2306.00978
- Official code: https://github.com/mit-han-lab/llm-awq
- HF AWQ docs: https://huggingface.co/docs/transformers/quantization/awq

This notebook uses AutoAWQ because it is the most convenient Colab API for producing a saved AWQ checkpoint. AutoAWQ may pin/downgrade `transformers`; run it in a fresh runtime.

In [ ]:
!pip -q install autoawq accelerate datasets sentencepiece safetensors

In [ ]:

import json
import os
import time
from pathlib import Path

import torch
from transformers import AutoTokenizer
from awq import AutoAWQForCausalLM


def bytes_to_gib(n):
    return n / (1024 ** 3)


def directory_size_bytes(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())


def cuda_peak_gib():
    if not torch.cuda.is_available():
        return 0.0
    return bytes_to_gib(torch.cuda.max_memory_allocated())


def infer_model_device(model):
    for candidate in (model, getattr(model, "model", None)):
        if candidate is None:
            continue
        try:
            return next(candidate.parameters()).device
        except (AttributeError, StopIteration):
            pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def generate_benchmark(model, tokenizer, prompt, max_new_tokens=64, runs=3):
    device = infer_model_device(model)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=8, do_sample=False)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    times = []
    token_count = 0
    with torch.no_grad():
        for _ in range(runs):
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            start = time.perf_counter()
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - start
            times.append(elapsed)
            token_count = out.shape[-1] - inputs["input_ids"].shape[-1]
    avg = sum(times) / len(times)
    return {
        "avg_seconds": avg,
        "generated_tokens": int(token_count),
        "tokens_per_second": float(token_count / avg) if avg else 0.0,
        "cuda_peak_gib": cuda_peak_gib(),
    }


def save_metrics(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(json.dumps(payload, indent=2))


In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = Path("/content/quant_outputs/awq_int4")
PROMPT = "Quantization reduces inference memory by"
MAX_NEW_TOKENS = 64
RUNS = 3

QUANT_CONFIG = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM",
}

CALIBRATION_TEXTS = [
    "Activation aware quantization chooses scales using activation statistics.",
    "A fair benchmark keeps the model, prompt, hardware, and dataset identical.",
    "Weight only int4 inference can reduce memory traffic for transformer layers.",
    "Calibration examples should be representative but do not require labels.",
] * 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoAWQForCausalLM.from_pretrained(
    MODEL_ID,
    low_cpu_mem_usage=True,
    use_cache=False,
    safetensors=True,
)
model.quantize(
    tokenizer,
    quant_config=QUANT_CONFIG,
    calib_data=CALIBRATION_TEXTS,
    max_calib_seq_len=512,
    n_parallel_calib_samples=4,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_quantized(OUTPUT_DIR / "model", safetensors=True)
tokenizer.save_pretrained(OUTPUT_DIR / "model")

model = AutoAWQForCausalLM.from_quantized(
    OUTPUT_DIR / "model",
    fuse_layers=True,
    safetensors=True,
    device_map="auto",
)
metrics = generate_benchmark(model, tokenizer, PROMPT, MAX_NEW_TOKENS, RUNS)
metrics.update({
    "method": "awq",
    "quant_config": QUANT_CONFIG,
    "model_id": MODEL_ID,
    "artifact_dir": str(OUTPUT_DIR / "model"),
    "artifact_size_gib": bytes_to_gib(directory_size_bytes(OUTPUT_DIR / "model")),
})
save_metrics(OUTPUT_DIR / "metrics.json", metrics)